# Quantum Attention — TREC-6 (Question-Type Classification, 6 classes)

Single-head and multi-head **quantum-scored attention** vs classical dot-product
attention, with a separability-grounded **entanglement ablation**
(`none / intra / cross / full`), on **TREC**.

This notebook runs four experiments and **checkpoints every result to Google Drive**
after each (condition, seed) run, so it survives Colab disconnects — just re-run the
cell and it resumes where it stopped.

**Experiments**
1. **A — Architecture ablation (4 heads):** quantum head as 1 of 4; the realistic setting.
2. **B — Single-head ablation:** quantum head is the *only* attention → tests whether
   entanglement matters once *undiluted*.
3. **C — Data-efficiency:** accuracy vs training size (the low-data regime where quantum
   methods are most defensible).
4. **D — Qubit width / encoding bottleneck:** 2 vs 4 vs 8 qubits (1/2/4 angles per side)
   → tests whether a wider encoding lets entanglement matter.

**Run order:** Setup → Download → Core → Config → Harness → run any Experiment cell(s)
→ Reporting. Recommended: **A and B first** (fast, answer the core question), then C and D.




## 1 · Setup (install, mount Drive, choose device)

In [ ]:
pip -q install -U pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 98.4 MB/s eta 0:00:00


In [ ]:
# PennyLane is the quantum simulator; torch ships with Colab.
#!pip -q install -U pennylane
import os, json, time, math, itertools
import torch
print("torch", torch.__version__)
import pennylane as qml
print("pennylane", qml.__version__)

# Mount Google Drive for checkpointing.
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'trec'
DRIVE_ROOT = '/content/drive/MyDrive/quantum_attention'
# --- THIS notebook (block-1 / depth) writes to its OWN folder so it can never ---
# --- overwrite the original block-0 results saved by the first notebook.       ---
RUN_TAG = 'block1depth'
RESULTS_DIR = os.path.join(DRIVE_ROOT, DATASET + '_' + RUN_TAG)   # .../quantum_attention/trec_block1depth
ORIG_DIR    = os.path.join(DRIVE_ROOT, DATASET)                   # original block-0 results: READ-ONLY here
FIG_DIR = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
print("This notebook writes ->", RESULTS_DIR)
print("Original block-0 results (read-only) <-", ORIG_DIR)

# The 4-qubit statevector sim is CPU-bound; GPU does not accelerate default.qubit.
# CPU is the right choice and avoids device-mismatch. (High-RAM runtime helps memory.)
DEVICE = 'cpu'
print("device =", DEVICE)


torch 2.11.0+cpu
pennylane 0.45.1
Mounted at /content/drive
This notebook writes -> /content/drive/MyDrive/quantum_attention/trec_block1depth
Original block-0 results (read-only) <- /content/drive/MyDrive/quantum_attention/trec
device = cpu


## 2 · Download data (cached on the Colab VM)

In [ ]:
import urllib.request
def download(url, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print("exists:", path); return
    urllib.request.urlretrieve(url, path)
    print("downloaded:", path, os.path.getsize(path), "bytes")

download("https://raw.githubusercontent.com/honnibal/dsr16_nlp/master/data/question_classification/train_5500.label", "trec/train.label")
download("https://raw.githubusercontent.com/justindomingue/nlp/master/question_classification/data/TREC_10.label", "trec/test.label")


downloaded: trec/train.label 335858 bytes
downloaded: trec/test.label 23354 bytes


## 3 · Core code (model + utilities)
Generalized quantum attention head (configurable qubit count), the small Transformer, data utilities, val-based training, and the permutation test.

In [ ]:
"""
qatt_core.py — Consolidated core for the Quantum Attention experiments.
Embedded verbatim into each Colab notebook. Generalized to support a
configurable number of qubits (even; half encode the query, half the key).

Contents:
  * QuantumAttentionHead / MixedMultiHeadAttention / TransformerBlock / TextTransformer
  * attention_entropy
  * tokenize / build_vocab / batchify
  * run_one (val-based model selection) + permutation_test + mean_std
The dataset loaders live in the notebook (they differ per dataset).
"""

import math, time, itertools
import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml

PAD, UNK = 0, 1


# =========================================================================== #
# Quantum scoring circuit (generalized to n_qubits, half query / half key)
# =========================================================================== #
def build_quantum_score_qnode(n_qubits=4, entangle="full"):
    """Wires 0..h-1 carry query angles, wires h..2h-1 carry key angles (h=n_qubits/2).
    weights shape (2, n_qubits, 3): rotation layer 0 (pre-entangler) + layer 1 (post).
    The post-entangler rotation layer is REQUIRED so CZ gates (diagonal in Z) become
    visible to the Z-basis readout; without it entanglement has zero effect.

    entangle:
      none  -> no CZ                          (score separable: <Z_q0 Z_k0>=<Z_q0><Z_k0>)
      intra -> CZ within query / within key   (still separable for this readout)
      cross -> CZ pairing q_i with k_i         (genuine query<->key entanglement)
      full  -> cross + intra
    """
    assert n_qubits % 2 == 0 and n_qubits >= 2
    h = n_qubits // 2
    q_wires = list(range(h))
    k_wires = list(range(h, n_qubits))
    cross = [(q_wires[i], k_wires[i]) for i in range(h)]
    intra = ([(q_wires[i], q_wires[i + 1]) for i in range(h - 1)]
             + [(k_wires[i], k_wires[i + 1]) for i in range(h - 1)])
    cz_pairs = {"none": [], "intra": intra,
                "cross": cross, "full": cross + intra}[entangle]
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        for w in range(n_qubits):
            qml.RX(weights[0, w, 0], wires=w)
            qml.RY(weights[0, w, 1], wires=w)
            qml.RZ(weights[0, w, 2], wires=w)
        for a, b in cz_pairs:
            qml.CZ(wires=[a, b])
        for w in range(n_qubits):
            qml.RX(weights[1, w, 0], wires=w)
            qml.RY(weights[1, w, 1], wires=w)
            qml.RZ(weights[1, w, 2], wires=w)
        return qml.expval(qml.PauliZ(q_wires[0]) @ qml.PauliZ(k_wires[0]))

    return circuit


class QuantumAttentionHead(nn.Module):
    def __init__(self, head_dim, n_qubits=4, entangle="full", chunk=8192):
        super().__init__()
        assert n_qubits % 2 == 0
        self.n_qubits = n_qubits
        self.h = n_qubits // 2                       # angles encoded per side
        self.entangle = entangle
        self.chunk = chunk                           # max pairs per circuit call
        self.q_proj = nn.Linear(head_dim, self.h)
        self.k_proj = nn.Linear(head_dim, self.h)
        self.qweights = nn.Parameter(0.1 * torch.randn(2, n_qubits, 3))
        self.scale = nn.Parameter(torch.tensor(4.0))
        self.circuit = build_quantum_score_qnode(n_qubits, entangle)

    def _run_circuit(self, flat, qw):
        # Evaluate in chunks so peak memory stays bounded (matters at n_qubits>=6,
        # where the statevector is 2^n_qubits per pair). Each chunk is autograd-
        # connected, so gradients still flow and accumulate into qw.
        n = flat.shape[0]
        if n <= self.chunk:
            return self.circuit(flat, qw)
        outs = [self.circuit(flat[i:i + self.chunk], qw)
                for i in range(0, n, self.chunk)]
        return torch.cat(outs, dim=0)

    def forward(self, q, k, v, key_mask=None, debug=False):
        B, L, D = q.shape
        qa = torch.tanh(self.q_proj(q)) * math.pi     # (B,L,h)
        ka = torch.tanh(self.k_proj(k)) * math.pi
        qa_exp = qa.unsqueeze(2).expand(B, L, L, self.h)
        ka_exp = ka.unsqueeze(1).expand(B, L, L, self.h)
        pairs = torch.cat([qa_exp, ka_exp], dim=-1)   # (B,L,L,n_qubits)
        flat = pairs.reshape(-1, self.n_qubits).float()
        # default.qubit simulates on CPU; run there and move scalars back.
        scores = self._run_circuit(flat.cpu(), self.qweights.cpu())
        scores = scores.reshape(B, L, L).to(device=v.device, dtype=v.dtype)
        if debug:
            with torch.no_grad():
                print(f"[qhead] score range [{scores.min():.3f},{scores.max():.3f}] "
                      f"std={scores.std():.3f}")
        scores = scores * self.scale
        if key_mask is not None:
            scores = scores.masked_fill(key_mask == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        return out, attn

    def self_test(self, device="cpu"):
        x = torch.randn(2, 3, self.q_proj.in_features, device=device)
        self.to(device)
        out, attn = self.forward(x, x, x, debug=True)
        out.sum().backward()
        g = self.qweights.grad
        gn = None if g is None else g.norm().item()
        ok = gn and gn > 1e-8
        print(f"[self_test] nq={self.n_qubits} grad_norm={gn} "
              f"{'OK' if ok else 'DEAD GRADIENT'}")
        self.zero_grad()
        return ok


class MixedMultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, use_quantum=True, entangle="full", n_qubits=4):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads = d_model, n_heads
        self.head_dim = d_model // n_heads
        self.use_quantum = use_quantum
        self.q_lin = nn.Linear(d_model, d_model)
        self.k_lin = nn.Linear(d_model, d_model)
        self.v_lin = nn.Linear(d_model, d_model)
        self.out_lin = nn.Linear(d_model, d_model)
        if use_quantum:
            self.qhead = QuantumAttentionHead(self.head_dim, n_qubits, entangle)

    def forward(self, x, key_mask=None, return_attn=False, debug=False):
        B, L, _ = x.shape
        H, Dh = self.n_heads, self.head_dim
        q = self.q_lin(x).view(B, L, H, Dh).transpose(1, 2)
        k = self.k_lin(x).view(B, L, H, Dh).transpose(1, 2)
        v = self.v_lin(x).view(B, L, H, Dh).transpose(1, 2)
        outs, attns = [], {}
        for hh in range(H):
            if self.use_quantum and hh == 0:
                o, a = self.qhead(q[:, hh], k[:, hh], v[:, hh],
                                  key_mask=key_mask, debug=debug)
                attns["quantum"] = a
            else:
                sc = torch.matmul(q[:, hh], k[:, hh].transpose(-2, -1)) / math.sqrt(Dh)
                if key_mask is not None:
                    sc = sc.masked_fill(key_mask == 0, float("-inf"))
                a = F.softmax(sc, dim=-1)
                o = torch.matmul(a, v[:, hh])
                if hh == 1 or (not self.use_quantum and hh == 0):
                    attns.setdefault("classical", a)
            outs.append(o)
        out = torch.stack(outs, dim=1).transpose(1, 2).reshape(B, L, H * Dh)
        out = self.out_lin(out)
        return (out, attns) if return_attn else out


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, use_quantum=False,
                 entangle="full", n_qubits=4, dropout=0.1):
        super().__init__()
        self.attn = MixedMultiHeadAttention(d_model, n_heads, use_quantum,
                                            entangle, n_qubits)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, ff_dim), nn.GELU(),
                                nn.Linear(ff_dim, d_model))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_mask=None, return_attn=False, debug=False):
        if return_attn:
            a_out, attns = self.attn(x, key_mask, return_attn=True, debug=debug)
        else:
            a_out, attns = self.attn(x, key_mask, debug=debug), None
        x = self.norm1(x + self.drop(a_out))
        x = self.norm2(x + self.drop(self.ff(x)))
        return (x, attns) if return_attn else x


class TextTransformer(nn.Module):
    def __init__(self, vocab_size, n_classes, d_model=64, n_heads=4, ff_dim=128,
                 max_len=40, use_quantum=True, entangle="full", n_qubits=4,
                 quantum_in_block=0, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.max_len = max_len
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ff_dim,
                             use_quantum=(use_quantum and b == quantum_in_block),
                             entangle=entangle, n_qubits=n_qubits, dropout=dropout)
            for b in range(2)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, input_ids, attn_mask, return_attn=False, debug=False):
        B, L = input_ids.shape
        L = min(L, self.max_len)
        input_ids = input_ids[:, :L]
        attn_mask = attn_mask[:, :L]
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)
        key_mask = attn_mask.unsqueeze(1)
        collected = {}
        for bi, blk in enumerate(self.blocks):
            if return_attn:
                x, attns = blk(x, key_mask, return_attn=True, debug=(debug and bi == 0))
                if attns:
                    collected[f"block{bi}"] = attns
            else:
                x = blk(x, key_mask, debug=(debug and bi == 0))
        x = self.norm(x)
        m = attn_mask.unsqueeze(-1).float()
        pooled = (x * m).sum(1) / m.sum(1).clamp(min=1.0)
        logits = self.head(pooled)
        return (logits, collected) if return_attn else logits


def attention_entropy(attn, key_mask=None, eps=1e-9):
    p = attn.clamp(min=eps)
    ent = -(p * p.log()).sum(-1)
    if key_mask is not None:
        valid = key_mask.squeeze(1).float()
        return ((ent * valid).sum() / valid.sum().clamp(min=1.0)).item()
    return ent.mean().item()


# =========================================================================== #
# Data utilities
# =========================================================================== #
def tokenize(s):
    return s.lower().split()


def build_vocab(examples, min_freq=2):
    from collections import Counter
    c = Counter()
    for e in examples:
        c.update(tokenize(e["text"]))
    v = {"<pad>": PAD, "<unk>": UNK}
    for w, f in c.most_common():
        if f >= min_freq:
            v[w] = len(v)
    return v


def encode(s, v, ml):
    return [v.get(t, UNK) for t in tokenize(s)][:ml]


def batchify(examples, v, ml, bs, shuffle, seed, device):
    idx = list(range(len(examples)))
    if shuffle:
        g = torch.Generator().manual_seed(seed)
        idx = torch.randperm(len(examples), generator=g).tolist()
    batches = []
    for i in range(0, len(examples), bs):
        chunk = [examples[j] for j in idx[i:i + bs]]
        seqs = [encode(e["text"], v, ml) for e in chunk]
        L = max(max((len(s) for s in seqs), default=1), 1)
        ids = torch.full((len(seqs), L), PAD, dtype=torch.long)
        m = torch.zeros((len(seqs), L), dtype=torch.long)
        for r, s in enumerate(seqs):
            if s:
                ids[r, :len(s)] = torch.tensor(s)
                m[r, :len(s)] = 1
        y = torch.tensor([e["label"] for e in chunk])
        batches.append((ids.to(device), m.to(device), y.to(device)))
    return batches


@torch.no_grad()
def accuracy(model, batches):
    model.eval()
    c = t = 0
    for ids, m, y in batches:
        c += (model(ids, m).argmax(-1) == y).sum().item()
        t += y.numel()
    return c / max(t, 1)


@torch.no_grad()
def mean_entropy(model, batches, max_batches=6):
    model.eval()
    q, cl = [], []
    for bi, (ids, m, y) in enumerate(batches):
        if bi >= max_batches:
            break
        _, attns = model(ids, m, return_attn=True)
        km = m[:, :ids.shape[1]].unsqueeze(1)
        # read entropy from whichever block holds the quantum head (any depth)
        qblk = next((d for d in attns.values() if "quantum" in d), {})
        if "quantum" in qblk:
            q.append(attention_entropy(qblk["quantum"], km))
        cblk = qblk if "classical" in qblk else \
               next((d for d in attns.values() if "classical" in d), {})
        if "classical" in cblk:
            cl.append(attention_entropy(cblk["classical"], km))
    return (sum(q) / len(q) if q else float("nan"),
            sum(cl) / len(cl) if cl else float("nan"))


# =========================================================================== #
# Train one (condition, seed) with val-based model selection
# =========================================================================== #
CONDITIONS = {
    "classical": dict(use_quantum=False, entangle="full"),
    "qnone":     dict(use_quantum=True,  entangle="none"),
    "qintra":    dict(use_quantum=True,  entangle="intra"),
    "qcross":    dict(use_quantum=True,  entangle="cross"),
    "qfull":     dict(use_quantum=True,  entangle="full"),
}


def run_one(cond, seed, data, vocab, n_classes, cfg):
    """cfg: dict with epochs, max_len, batch_size, d_model, n_heads, n_qubits,
    lr, qlr, device."""
    train_fit, val, test = data
    cc = CONDITIONS[cond]
    device = cfg["device"]
    torch.manual_seed(seed)
    model = TextTransformer(
        vocab_size=len(vocab), n_classes=n_classes,
        d_model=cfg["d_model"], n_heads=cfg["n_heads"], max_len=cfg["max_len"],
        use_quantum=cc["use_quantum"], entangle=cc["entangle"],
        n_qubits=cfg["n_qubits"],
        quantum_in_block=cfg.get("quantum_in_block", 0)).to(device)
    qp = [p for n, p in model.named_parameters() if "qweights" in n]
    cp = [p for n, p in model.named_parameters() if "qweights" not in n]
    groups = [{"params": cp, "lr": cfg["lr"]}]
    if qp:
        groups.append({"params": qp, "lr": cfg["qlr"]})
    opt = torch.optim.Adam(groups)
    val_b = batchify(val, vocab, cfg["max_len"], 128, False, 0, device)
    test_b = batchify(test, vocab, cfg["max_len"], 128, False, 0, device)
    best_val, test_at_best, best_ep = -1.0, 0.0, 0
    t0 = time.time()
    for ep in range(1, cfg["epochs"] + 1):
        model.train()
        for ids, m, y in batchify(train_fit, vocab, cfg["max_len"],
                                  cfg["batch_size"], True, 1000 * seed + ep, device):
            opt.zero_grad()
            F.cross_entropy(model(ids, m), y).backward()
            opt.step()
        va = accuracy(model, val_b)
        if va > best_val:
            best_val, test_at_best, best_ep = va, accuracy(model, test_b), ep
    h_q, h_c = mean_entropy(model, test_b)
    return dict(test_acc=test_at_best, val_acc=best_val, best_epoch=best_ep,
                H_quantum=h_q, H_classical=h_c,
                params=sum(p.numel() for p in model.parameters() if p.requires_grad),
                n_qubits=cfg["n_qubits"], n_heads=cfg["n_heads"],
                quantum_in_block=cfg.get("quantum_in_block", 0),
                seconds=round(time.time() - t0, 1))


def mean_std(xs):
    n = len(xs)
    mu = sum(xs) / n
    sd = (sum((x - mu) ** 2 for x in xs) / (n - 1)) ** 0.5 if n > 1 else 0.0
    return mu, sd


def permutation_test(a, b, max_exact=200000, n_random=100000, seed=0):
    a, b = list(a), list(b)
    na, nb = len(a), len(b)
    pooled = a + b
    obs = abs(sum(a) / na - sum(b) / nb)
    total = math.comb(na + nb, na)
    cnt = 0
    if total <= max_exact:
        for combo in itertools.combinations(range(na + nb), na):
            s = set(combo)
            ga = [pooled[i] for i in combo]
            gb = [pooled[i] for i in range(na + nb) if i not in s]
            if abs(sum(ga) / na - sum(gb) / nb) >= obs - 1e-12:
                cnt += 1
        return obs, cnt / total, f"exact ({total})"
    rng = torch.Generator().manual_seed(seed)
    for _ in range(n_random):
        perm = torch.randperm(na + nb, generator=rng).tolist()
        ga = [pooled[i] for i in perm[:na]]
        gb = [pooled[i] for i in perm[na:]]
        if abs(sum(ga) / na - sum(gb) / nb) >= obs - 1e-12:
            cnt += 1
    return obs, cnt / n_random, f"sampled ({n_random})"


## 4 · Dataset config + load + fixed split

In [ ]:

COARSE = ["ABBR","ENTY","DESC","HUM","LOC","NUM"]
C2I = {c:i for i,c in enumerate(COARSE)}
def load_dataset_examples():
    def rd(p):
        ex=[]
        for line in open(p,"rb"):
            line=line.replace(b"\xf0",b" ").strip().decode("latin-1")
            if not line: continue
            lab,_,t=line.partition(" "); c=lab.split(":")[0]
            if c in C2I and t: ex.append({"text":t,"label":C2I[c]})
        return ex
    return rd("trec/train.label"), rd("trec/test.label"), COARSE

# ---- experiment configuration ----
N_CLASSES   = 6
MAX_LEN     = 20
DEFAULT_N   = 2000      # train-fit size for experiments A, B, D
EPOCHS      = 20
VAL_SIZE    = 500
MIN_FREQ    = 2
SEEDS       = [0, 1, 2, 3, 4]       # set to [0,1,2] for a quick run
N_EFF       = [100, 250, 500, 1000, 2000]          # training sizes for the data-efficiency sweep

TRAIN_ALL, TEST_ALL, LABELS = load_dataset_examples()
assert len(LABELS) == N_CLASSES
print(f"{DATASET}: train={len(TRAIN_ALL)} test={len(TEST_ALL)} classes={N_CLASSES} {LABELS}")
from collections import Counter
print("label counts:", dict(Counter(e['label'] for e in TRAIN_ALL)))
_lens = sorted(len(tokenize(e['text'])) for e in TRAIN_ALL)
print(f"token length: median={_lens[len(_lens)//2]} p95={_lens[int(0.95*len(_lens))]} "
      f"max={_lens[-1]} | MAX_LEN={MAX_LEN}")


trec: train=5452 test=500 classes=6 ['ABBR', 'ENTY', 'DESC', 'HUM', 'LOC', 'NUM']
label counts: {2: 1162, 1: 1250, 0: 86, 3: 1223, 5: 896, 4: 835}
token length: median=10 p95=17 max=37 | MAX_LEN=20


## 5 · Checkpointed experiment harness
`run_experiment` saves to Drive after **every** (config, seed); re-running skips finished cells. Data/vocab are built per training size and cached.

In [ ]:
def _load(path):
    return json.load(open(path)) if os.path.exists(path) else {}

def _save(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    json.dump(obj, open(tmp, "w"), indent=2)
    os.replace(tmp, path)            # avoids half-written checkpoints on interrupt

_data_cache = {}
def make_split(limit_train):
    g = torch.Generator().manual_seed(12345)
    tr = list(TRAIN_ALL)
    perm = torch.randperm(len(tr), generator=g).tolist()
    tr = [tr[i] for i in perm]
    if limit_train is not None:
        tr = tr[:limit_train + VAL_SIZE]
    val = tr[:VAL_SIZE]; train_fit = tr[VAL_SIZE:]
    vocab = build_vocab(train_fit, MIN_FREQ)
    return (train_fit, val, TEST_ALL), vocab, N_CLASSES

def get_data(limit_train):
    if limit_train not in _data_cache:
        _data_cache[limit_train] = make_split(limit_train)
    return _data_cache[limit_train]

BASE = dict(epochs=EPOCHS, max_len=MAX_LEN, batch_size=64, d_model=64,
            n_heads=4, n_qubits=4, lr=2e-3, qlr=1e-2, device=DEVICE)
def cfg(cond, key, **over):
    c = dict(BASE); c.update(over); c["cond"] = cond; c["key"] = key
    c.setdefault("limit_train", DEFAULT_N); return c

def run_experiment(exp_name, grid, seeds):
    path = os.path.join(RESULTS_DIR, exp_name + ".json")
    results = _load(path)
    t_start = time.time()
    for c in grid:
        key = c["key"]
        data, vocab, nc = get_data(c["limit_train"])
        results.setdefault(key, {})
        for s in seeds:
            if str(s) in results[key]:
                print(f"  skip {key} seed{s} (test={results[key][str(s)]['test_acc']:.4f})")
                continue
            print(f"  run  {key} seed{s} ...", flush=True)
            r = run_one(c["cond"], s, data, vocab, nc, c)
            results[key][str(s)] = r
            _save(path, results)
            print(f"       test={r['test_acc']:.4f} val={r['val_acc']:.4f} "
                  f"ep*={r['best_epoch']} H_q={r['H_quantum']:.3f} "
                  f"H_c={r['H_classical']:.3f} ({r['seconds']}s)", flush=True)
    print(f"[done] {exp_name}: {time.time()-t_start:.0f}s total -> {path}")
    return results


## Experiment A — Architecture ablation (4 heads)
The quantum head is **1 of 4** attention heads (the realistic setting). All five conditions share the same architecture; only the entangling-gate set changes.

In [ ]:
gridA = [cfg(c, f"A_{c}") for c in
         ["classical","qnone","qintra","qcross","qfull"]]
run_experiment("expA_4head", gridA, SEEDS)


  skip A_classical seed0 (test=0.8100)
  skip A_classical seed1 (test=0.8400)
  skip A_classical seed2 (test=0.8300)
  skip A_classical seed3 (test=0.8320)
  skip A_classical seed4 (test=0.8140)
  skip A_qnone seed0 (test=0.8420)
  skip A_qnone seed1 (test=0.8380)
  skip A_qnone seed2 (test=0.7980)
  skip A_qnone seed3 (test=0.8160)
  skip A_qnone seed4 (test=0.8060)
  skip A_qintra seed0 (test=0.8340)
  skip A_qintra seed1 (test=0.8200)
  skip A_qintra seed2 (test=0.8340)
  skip A_qintra seed3 (test=0.8080)
  skip A_qintra seed4 (test=0.8260)
  skip A_qcross seed0 (test=0.8460)
  skip A_qcross seed1 (test=0.8260)
  skip A_qcross seed2 (test=0.8100)
  skip A_qcross seed3 (test=0.8140)
  skip A_qcross seed4 (test=0.8140)
  skip A_qfull seed0 (test=0.8500)
  skip A_qfull seed1 (test=0.8180)
  skip A_qfull seed2 (test=0.8100)
  skip A_qfull seed3 (test=0.8280)
  skip A_qfull seed4 (test=0.8060)
[done] expA_4head: 0s total -> /content/drive/MyDrive/quantum_attention/trec_block1depth/expA_4

{'A_classical': {'0': {'test_acc': 0.81,
   'val_acc': 0.766,
   'best_epoch': 8,
   'H_quantum': nan,
   'H_classical': 1.0385698676109314,
   'params': 168710,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 0,
   'seconds': 19.1},
  '1': {'test_acc': 0.84,
   'val_acc': 0.764,
   'best_epoch': 13,
   'H_quantum': nan,
   'H_classical': 0.6480968743562698,
   'params': 168710,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 0,
   'seconds': 18.7},
  '2': {'test_acc': 0.83,
   'val_acc': 0.756,
   'best_epoch': 16,
   'H_quantum': nan,
   'H_classical': 0.8995702564716339,
   'params': 168710,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 0,
   'seconds': 17.9},
  '3': {'test_acc': 0.832,
   'val_acc': 0.786,
   'best_epoch': 9,
   'H_quantum': nan,
   'H_classical': 0.8594456017017365,
   'params': 168710,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 0,
   'seconds': 18.0},
  '4': {'test_acc': 0.814,
   'val_acc': 0.758,
   'best_epoch': 1

## Experiment B — Single-head ablation (dilution test)
`n_heads=1`: the quantum circuit is the **only** attention mechanism. If entanglement matters at all, it should be most visible here, undiluted by parallel classical heads.

In [ ]:
gridB = [cfg(c, f"B_{c}", n_heads=1) for c in
         ["classical","qnone","qintra","qcross","qfull"]]
run_experiment("expB_1head", gridB, SEEDS)


  skip B_classical seed0 (test=0.8400)
  skip B_classical seed1 (test=0.8360)
  skip B_classical seed2 (test=0.8200)
  skip B_classical seed3 (test=0.8080)
  skip B_classical seed4 (test=0.8240)
  skip B_qnone seed0 (test=0.8400)
  skip B_qnone seed1 (test=0.8440)
  skip B_qnone seed2 (test=0.8340)
  skip B_qnone seed3 (test=0.8400)
  skip B_qnone seed4 (test=0.8280)
  skip B_qintra seed0 (test=0.8240)
  skip B_qintra seed1 (test=0.8240)
  skip B_qintra seed2 (test=0.8320)
  skip B_qintra seed3 (test=0.8380)
  skip B_qintra seed4 (test=0.8320)
  skip B_qcross seed0 (test=0.8120)
  skip B_qcross seed1 (test=0.8240)
  skip B_qcross seed2 (test=0.8280)
  skip B_qcross seed3 (test=0.8100)
  skip B_qcross seed4 (test=0.8380)
  skip B_qfull seed0 (test=0.8300)
  skip B_qfull seed1 (test=0.8280)
  skip B_qfull seed2 (test=0.8000)
  skip B_qfull seed3 (test=0.8060)
  skip B_qfull seed4 (test=0.8160)
[done] expB_1head: 0s total -> /content/drive/MyDrive/quantum_attention/trec_block1depth/expB_1

{'B_classical': {'0': {'test_acc': 0.84,
   'val_acc': 0.764,
   'best_epoch': 12,
   'H_quantum': nan,
   'H_classical': 0.6311395466327667,
   'params': 168710,
   'n_qubits': 4,
   'n_heads': 1,
   'quantum_in_block': 0,
   'seconds': 14.4},
  '1': {'test_acc': 0.836,
   'val_acc': 0.756,
   'best_epoch': 11,
   'H_quantum': nan,
   'H_classical': 0.7711444646120071,
   'params': 168710,
   'n_qubits': 4,
   'n_heads': 1,
   'quantum_in_block': 0,
   'seconds': 14.4},
  '2': {'test_acc': 0.82,
   'val_acc': 0.762,
   'best_epoch': 16,
   'H_quantum': nan,
   'H_classical': 0.7240406274795532,
   'params': 168710,
   'n_qubits': 4,
   'n_heads': 1,
   'quantum_in_block': 0,
   'seconds': 14.5},
  '3': {'test_acc': 0.808,
   'val_acc': 0.774,
   'best_epoch': 11,
   'H_quantum': nan,
   'H_classical': 0.920240581035614,
   'params': 168710,
   'n_qubits': 4,
   'n_heads': 1,
   'quantum_in_block': 0,
   'seconds': 14.8},
  '4': {'test_acc': 0.824,
   'val_acc': 0.766,
   'best_epoch':

## Experiment C — Data-efficiency
Accuracy vs training size. Quantum methods are most defensible in the low-data regime, so this curve is where a positive effect is most likely to appear.

In [ ]:
gridC = [cfg(c, f"C_{c}_N{N}", limit_train=N)
         for N in N_EFF
         for c in ["classical","qnone","qcross","qfull"]]
run_experiment("expC_dataeff", gridC, SEEDS)


  skip C_classical_N100 seed0 (test=0.5840)
  skip C_classical_N100 seed1 (test=0.6220)
  skip C_classical_N100 seed2 (test=0.5560)
  skip C_classical_N100 seed3 (test=0.5660)
  skip C_classical_N100 seed4 (test=0.6120)
  skip C_qnone_N100 seed0 (test=0.5100)
  skip C_qnone_N100 seed1 (test=0.5500)
  skip C_qnone_N100 seed2 (test=0.5300)
  skip C_qnone_N100 seed3 (test=0.5900)
  skip C_qnone_N100 seed4 (test=0.5860)
  skip C_qcross_N100 seed0 (test=0.5120)
  skip C_qcross_N100 seed1 (test=0.5260)
  skip C_qcross_N100 seed2 (test=0.4300)
  skip C_qcross_N100 seed3 (test=0.5680)
  skip C_qcross_N100 seed4 (test=0.5500)
  skip C_qfull_N100 seed0 (test=0.5220)
  skip C_qfull_N100 seed1 (test=0.5320)
  skip C_qfull_N100 seed2 (test=0.4160)
  skip C_qfull_N100 seed3 (test=0.5240)
  skip C_qfull_N100 seed4 (test=0.5520)
  skip C_classical_N250 seed0 (test=0.6820)
  skip C_classical_N250 seed1 (test=0.6620)
  skip C_classical_N250 seed2 (test=0.6780)
  skip C_classical_N250 seed3 (test=0.6380)

{'C_classical_N100': {'0': {'test_acc': 0.584,
   'val_acc': 0.558,
   'best_epoch': 18,
   'H_quantum': nan,
   'H_classical': 1.0500523149967194,
   'params': 74182,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 0,
   'seconds': 1.9},
  '1': {'test_acc': 0.622,
   'val_acc': 0.554,
   'best_epoch': 18,
   'H_quantum': nan,
   'H_classical': 1.098571926355362,
   'params': 74182,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 0,
   'seconds': 1.9},
  '2': {'test_acc': 0.556,
   'val_acc': 0.492,
   'best_epoch': 18,
   'H_quantum': nan,
   'H_classical': 1.0993587970733643,
   'params': 74182,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 0,
   'seconds': 1.8},
  '3': {'test_acc': 0.566,
   'val_acc': 0.554,
   'best_epoch': 14,
   'H_quantum': nan,
   'H_classical': 1.2443737983703613,
   'params': 74182,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 0,
   'seconds': 1.8},
  '4': {'test_acc': 0.612,
   'val_acc': 0.516,
   'best_epoch': 

## Experiment D — Qubit width / encoding bottleneck
2 / 4 / 8 qubits = 1 / 2 / 4 angles encoded per side. Tests whether widening the input encoding gives entanglement room to matter. **Memory note:** the statevector is 2^n_qubits per pair, so this cell uses a smaller batch and sequence length; n_qubits=8 is the slowest run in the notebook.

In [ ]:
# Conservative settings keep n_qubits=8 within memory; classical is the reference.
# n_qubits=8 is the slowest cell in the whole study (statevector = 2^8 per pair),
# so this experiment uses a smaller subset, fewer epochs, and fewer seeds.
# The grid runs nq=2 and nq=4 first (cheap) and checkpoints, so you get those
# results before nq=8 starts — you can interrupt before nq=8 if short on time.
D_OVER  = dict(batch_size=16, max_len=min(MAX_LEN, 14), limit_train=800, epochs=12)
D_SEEDS = SEEDS[:3]                         # 3 seeds for the expensive sweep
gridD = [cfg("classical", "D_classical", **D_OVER)]
for nq in [2, 4, 8]:                        # nq=8 last (slowest)
    for c in ["qnone","qcross","qfull"]:
        gridD.append(cfg(c, f"D_{c}_nq{nq}", n_qubits=nq, **D_OVER))
run_experiment("expD_qubits", gridD, D_SEEDS)


  run  D_classical seed0 ...
       test=0.7680 val=0.7340 ep*=9 H_q=nan H_c=0.712 (10.3s)
  run  D_classical seed1 ...
       test=0.7960 val=0.7460 ep*=11 H_q=nan H_c=0.804 (9.9s)
  run  D_classical seed2 ...
       test=0.7820 val=0.7360 ep*=8 H_q=nan H_c=0.851 (10.1s)
  run  D_qnone_nq2 seed0 ...
       test=0.7780 val=0.7280 ep*=11 H_q=1.662 H_c=0.619 (27.6s)
  run  D_qnone_nq2 seed1 ...
       test=0.7660 val=0.7160 ep*=7 H_q=0.879 H_c=0.652 (27.7s)
  run  D_qnone_nq2 seed2 ...
       test=0.7240 val=0.7420 ep*=12 H_q=1.789 H_c=0.824 (30.5s)
  run  D_qcross_nq2 seed0 ...
       test=0.7780 val=0.7200 ep*=12 H_q=1.561 H_c=0.719 (30.2s)
  run  D_qcross_nq2 seed1 ...
       test=0.8340 val=0.7200 ep*=11 H_q=1.584 H_c=0.608 (31.4s)
  run  D_qcross_nq2 seed2 ...
       test=0.7640 val=0.7400 ep*=10 H_q=1.228 H_c=0.887 (30.4s)
  run  D_qfull_nq2 seed0 ...
       test=0.7780 val=0.7200 ep*=12 H_q=1.561 H_c=0.719 (30.9s)
  run  D_qfull_nq2 seed1 ...
       test=0.8340 val=0.7200 ep*=11 H

{'D_classical': {'0': {'test_acc': 0.768,
   'val_acc': 0.734,
   'best_epoch': 9,
   'H_quantum': nan,
   'H_classical': 0.7119521498680115,
   'params': 109446,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 0,
   'seconds': 10.3},
  '1': {'test_acc': 0.796,
   'val_acc': 0.746,
   'best_epoch': 11,
   'H_quantum': nan,
   'H_classical': 0.8044213205575943,
   'params': 109446,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 0,
   'seconds': 9.9},
  '2': {'test_acc': 0.782,
   'val_acc': 0.736,
   'best_epoch': 8,
   'H_quantum': nan,
   'H_classical': 0.8513142466545105,
   'params': 109446,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 0,
   'seconds': 10.1}},
 'D_qnone_nq2': {'0': {'test_acc': 0.778,
   'val_acc': 0.728,
   'best_epoch': 11,
   'H_quantum': 1.6620756387710571,
   'H_classical': 0.6185663193464279,
   'params': 109493,
   'n_qubits': 2,
   'n_heads': 4,
   'quantum_in_block': 0,
   'seconds': 27.6},
  '1': {'test_acc': 0.766,
   'va

## Experiment E — Depth ablation (quantum head in the LAST block)
Identical to Experiment A in every respect **except** the quantum head sits in
block 1 (the final block; the model has 2 blocks) instead of block 0. This
isolates *layer position* as the only variable: same single quantum head, same
dose of entanglement, deeper features. Compare `E_*` against `A_*`.


In [ ]:
gridE = [cfg(c, f"E_{c}", quantum_in_block=1) for c in
         ["classical","qnone","qintra","qcross","qfull"]]
run_experiment("expE_depth", gridE, SEEDS)


  run  E_classical seed0 ...
       test=0.8100 val=0.7660 ep*=8 H_q=nan H_c=1.039 (18.8s)
  run  E_classical seed1 ...
       test=0.8400 val=0.7640 ep*=13 H_q=nan H_c=0.648 (19.6s)
  run  E_classical seed2 ...
       test=0.8300 val=0.7560 ep*=16 H_q=nan H_c=0.900 (19.7s)
  run  E_classical seed3 ...
       test=0.8320 val=0.7860 ep*=9 H_q=nan H_c=0.859 (20.7s)
  run  E_classical seed4 ...
       test=0.8140 val=0.7580 ep*=13 H_q=nan H_c=0.955 (18.7s)
  run  E_qnone seed0 ...
       test=0.8380 val=0.7660 ep*=16 H_q=1.202 H_c=1.096 (334.0s)
  run  E_qnone seed1 ...
       test=0.8240 val=0.7580 ep*=15 H_q=1.416 H_c=0.988 (343.5s)
  run  E_qnone seed2 ...
       test=0.8060 val=0.7660 ep*=11 H_q=1.670 H_c=1.152 (347.6s)
  run  E_qnone seed3 ...
       test=0.8140 val=0.7880 ep*=16 H_q=1.373 H_c=1.007 (361.5s)
  run  E_qnone seed4 ...
       test=0.8060 val=0.7500 ep*=9 H_q=1.609 H_c=0.932 (337.9s)
  run  E_qintra seed0 ...
       test=0.8420 val=0.7620 ep*=16 H_q=0.984 H_c=0.895 (402.

{'E_classical': {'0': {'test_acc': 0.81,
   'val_acc': 0.766,
   'best_epoch': 8,
   'H_quantum': nan,
   'H_classical': 1.0385698676109314,
   'params': 168710,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 1,
   'seconds': 18.8},
  '1': {'test_acc': 0.84,
   'val_acc': 0.764,
   'best_epoch': 13,
   'H_quantum': nan,
   'H_classical': 0.6480968743562698,
   'params': 168710,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 1,
   'seconds': 19.6},
  '2': {'test_acc': 0.83,
   'val_acc': 0.756,
   'best_epoch': 16,
   'H_quantum': nan,
   'H_classical': 0.8995702564716339,
   'params': 168710,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 1,
   'seconds': 19.7},
  '3': {'test_acc': 0.832,
   'val_acc': 0.786,
   'best_epoch': 9,
   'H_quantum': nan,
   'H_classical': 0.8594456017017365,
   'params': 168710,
   'n_qubits': 4,
   'n_heads': 4,
   'quantum_in_block': 1,
   'seconds': 20.7},
  '4': {'test_acc': 0.814,
   'val_acc': 0.758,
   'best_epoch': 1

## Reporting & figures
Reads the Drive checkpoints, prints mean±std tables with permutation tests, and saves publication figures (PNG + PDF) to the Drive `figures/` folder. Safe to run with partial results.

In [ ]:
import matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.rcParams.update({"font.size":11,"axes.spines.top":False,"axes.spines.right":False,
                     "axes.grid":True,"grid.alpha":0.25,"grid.linestyle":"--"})

def _accs(res, key):
    return [v["test_acc"] for v in res.get(key, {}).values()]

def _ms(xs):
    n=len(xs); mu=sum(xs)/n if n else float('nan')
    sd=(sum((x-mu)**2 for x in xs)/(n-1))**0.5 if n>1 else 0.0
    return mu,sd,n

def table(res, keys, title, pairs=None):
    print(f"\n=== {title} ===")
    print(f"{'key':16} {'mean':>8} {'std':>7} {'n':>3}  H_q     H_c")
    summ={}
    for k in keys:
        a=_accs(res,k)
        if not a: continue
        mu,sd,n=_ms(a); summ[k]=a
        runs=list(res[k].values())
        hq=sum(float(x['H_quantum']) if str(x['H_quantum'])!='nan' else float('nan') for x in runs)/len(runs)
        hc=sum(float(x['H_classical']) for x in runs)/len(runs)
        print(f"{k:16} {mu:8.4f} {sd:7.4f} {n:3d}  {hq:6.3f}  {hc:6.3f}")
    print(f"{'random':16} {1.0/N_CLASSES:8.4f}")
    if pairs:
        for x,y in pairs:
            if len(summ.get(x,[]))>1 and len(summ.get(y,[]))>1:
                _,p,mode=permutation_test(summ[x],summ[y])
                dm=sum(summ[x])/len(summ[x])-sum(summ[y])/len(summ[y])
                print(f"   {x} vs {y}: Δ={dm:+.4f} p={p:.4f} {'*' if p<0.05 else ''} [{mode}]")
    return summ

def bar(res, keys, labels, title, fname):
    ms=[_ms(_accs(res,k)) for k in keys]
    if not any(m[2] for m in ms): print("no data for",fname); return
    mu=[m[0] for m in ms]; sd=[m[1] for m in ms]
    cols=["#9aa0a6","#a8d0e6","#a8d0e6","#1b6ca8","#0a3d62"][:len(keys)]
    fig,ax=plt.subplots(figsize=(6.2,4.2)); x=range(len(keys))
    ax.bar(x,mu,yerr=sd,capsize=5,color=cols,edgecolor="black",linewidth=.6)
    ax.set_xticks(list(x)); ax.set_xticklabels(labels)
    ax.set_ylabel("test accuracy"); ax.set_title(title)
    lo=min(m-s for m,s in zip(mu,sd)); hi=max(m+s for m,s in zip(mu,sd))
    pad=(hi-lo)*.5+.005; ax.set_ylim(lo-pad,hi+pad*1.4)
    for xi,m in zip(x,mu): ax.text(xi,m,f"{m:.3f}",ha="center",va="bottom",fontsize=8)
    fig.tight_layout()
    for ext in ("png","pdf"): fig.savefig(os.path.join(FIG_DIR,f"{fname}.{ext}"),dpi=300,bbox_inches="tight")
    plt.close(fig); print("saved",fname)

def line_vs(res, xvals, key_fmt, conds, title, xlabel, fname):
    fig,ax=plt.subplots(figsize=(6.2,4.2))
    for c in conds:
        ys,es=[],[]
        for xv in xvals:
            mu,sd,n=_ms(_accs(res,key_fmt(c,xv))); ys.append(mu); es.append(sd)
        if any(y==y for y in ys):
            ax.errorbar(xvals,ys,yerr=es,marker="o",capsize=4,label=c)
    ax.set_xlabel(xlabel); ax.set_ylabel("test accuracy"); ax.set_title(title)
    ax.axhline(1.0/N_CLASSES,ls=":",color="#888",label="random")
    ax.legend(frameon=False,fontsize=9); fig.tight_layout()
    for ext in ("png","pdf"): fig.savefig(os.path.join(FIG_DIR,f"{fname}.{ext}"),dpi=300,bbox_inches="tight")
    plt.close(fig); print("saved",fname)

CONDS5=["classical","qnone","qintra","qcross","qfull"]
LAB5=["classical","q-none","q-intra","q-cross","q-full"]
ABL_PAIRS=[("{p}qfull","{p}classical"),("{p}qfull","{p}qnone"),
           ("{p}qcross","{p}qnone"),("{p}qfull","{p}qcross")]

# Experiment A
rA=_load(os.path.join(ORIG_DIR,"expA_4head.json"))
if rA:
    table(rA,[f"A_{c}" for c in CONDS5],"A · 4-head ablation",
          [(x.format(p="A_"),y.format(p="A_")) for x,y in ABL_PAIRS])
    bar(rA,[f"A_{c}" for c in CONDS5],LAB5,f"{DATASET.upper()} · 4-head ablation","A_4head")

# Experiment B
rB=_load(os.path.join(RESULTS_DIR,"expB_1head.json"))
if rB:
    table(rB,[f"B_{c}" for c in CONDS5],"B · single-head ablation",
          [(x.format(p="B_"),y.format(p="B_")) for x,y in ABL_PAIRS])
    bar(rB,[f"B_{c}" for c in CONDS5],LAB5,f"{DATASET.upper()} · single head","B_1head")

# Experiment C
rC=_load(os.path.join(RESULTS_DIR,"expC_dataeff.json"))
if rC:
    table(rC,[f"C_{c}_N{N}" for N in N_EFF for c in ["classical","qnone","qcross","qfull"]],
          "C · data-efficiency")
    line_vs(rC,N_EFF,lambda c,N:f"C_{c}_N{N}",["classical","qnone","qcross","qfull"],
            f"{DATASET.upper()} · data efficiency","train size N","C_dataeff")

# Experiment D
rD=_load(os.path.join(RESULTS_DIR,"expD_qubits.json"))
if rD:
    table(rD,["D_classical"]+[f"D_{c}_nq{nq}" for nq in [2,4,8] for c in ["qnone","qcross","qfull"]],
          "D · qubit width")
    line_vs(rD,[2,4,8],lambda c,nq:f"D_{c}_nq{nq}",["qnone","qcross","qfull"],
            f"{DATASET.upper()} · qubit width","n_qubits","D_qubits")

print("\nAll figures saved to", FIG_DIR)



=== A · 4-head ablation ===
key                  mean     std   n  H_q     H_c
A_classical        0.8236  0.0107   5     nan   0.871
A_qnone            0.8176  0.0071   5   1.430   0.974
A_qintra           0.8248  0.0119   5   1.319   0.901
A_qcross           0.8300  0.0141   5   1.145   0.911
A_qfull            0.8156  0.0114   5   1.295   0.947
random             0.1667
   A_qfull vs A_classical: Δ=-0.0080 p=0.3492  [exact (252)]
   A_qfull vs A_qnone: Δ=-0.0020 p=0.8095  [exact (252)]
   A_qcross vs A_qnone: Δ=+0.0124 p=0.1349  [exact (252)]
   A_qfull vs A_qcross: Δ=-0.0144 p=0.1190  [exact (252)]
saved A_4head

=== B · single-head ablation ===
key                  mean     std   n  H_q     H_c
B_classical        0.8256  0.0128   5     nan   0.755
B_qnone            0.8372  0.0063   5   1.661   0.749
B_qintra           0.8300  0.0060   5   1.758   0.776
B_qcross           0.8224  0.0116   5   1.732   0.800
B_qfull            0.8160  0.0132   5   1.710   0.767
random             0.

In [ ]:
# Experiment E — depth ablation reporting
rE=_load(os.path.join(RESULTS_DIR,"expE_depth.json"))
if rE:
    table(rE,[f"E_{c}" for c in CONDS5],"E · quantum head in LAST block",
          [(x.format(p="E_"),y.format(p="E_")) for x,y in ABL_PAIRS])
    bar(rE,[f"E_{c}" for c in CONDS5],LAB5,f"{DATASET.upper()} · quantum in last block","E_depth")

# Cross-experiment: does layer position move accuracy?  (A = block 0, E = block 1)
rA_=_load(os.path.join(ORIG_DIR,"expA_4head.json"))
if rA_ and rE:
    print("\n=== Layer-position effect: block 0 (A) vs block 1 (E) ===")
    for c in CONDS5:
        a=_accs(rA_,f"A_{c}"); e=_accs(rE,f"E_{c}")
        if len(a)>1 and len(e)>1:
            _,p,mode=permutation_test(a,e)
            d=sum(e)/len(e)-sum(a)/len(a)
            print(f"   {c:10} block1-block0 Δ={d:+.4f} p={p:.4f} {'*' if p<0.05 else ''} [{mode}]")



=== E · quantum head in LAST block ===
key                  mean     std   n  H_q     H_c
E_classical        0.8252  0.0127   5     nan   0.880
E_qnone            0.8176  0.0136   5   1.454   1.035
E_qintra           0.8292  0.0178   5   1.399   1.005
E_qcross           0.8356  0.0145   5   1.303   1.026
E_qfull            0.8288  0.0132   5   1.460   1.055
random             0.1667
   E_qfull vs E_classical: Δ=+0.0036 p=0.7143  [exact (252)]
   E_qfull vs E_qnone: Δ=+0.0112 p=0.2460  [exact (252)]
   E_qcross vs E_qnone: Δ=+0.0180 p=0.0873  [exact (252)]
   E_qfull vs E_qcross: Δ=-0.0068 p=0.4762  [exact (252)]
saved E_depth

=== Layer-position effect: block 0 (A) vs block 1 (E) ===
   classical  block1-block0 Δ=+0.0016 p=0.8889  [exact (252)]
   qnone      block1-block0 Δ=+0.0000 p=1.0000  [exact (252)]
   qintra     block1-block0 Δ=+0.0044 p=0.6825  [exact (252)]
   qcross     block1-block0 Δ=+0.0056 p=0.5714  [exact (252)]
   qfull      block1-block0 Δ=+0.0132 p=0.1349  [exact (25